In [1]:
pip install reportlab

Note: you may need to restart the kernel to use updated packages.


In [29]:
"""
Synthetic SOP PDF Generator v2

This script generates a more diverse synthetic GMP-style SOP corpus
for research on document accessibility, parsing, and retrieval systems.

Improvements over v1:
- SOP type-specific templates
- More diverse titles
- Type-specific materials and procedure steps
- Variable parameters (time, pH, temperature, rpm, etc.)
- Reduced near-duplicate documents

Author: Sora Kim
Project: SOP Accessibility System
"""

import os
import random
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# --------------------------------------------------
# Configuration
# --------------------------------------------------

OUTPUT_DIR = "synthetic_sops/qc"
NUM_FILES = 500

# --------------------------------------------------
# Global text pools
# --------------------------------------------------

PURPOSE_LINES = {
    "sterility": [
        "This procedure describes the method for sterility testing of aseptic samples.",
        "This procedure defines the steps required to perform sterility testing in the QC microbiology laboratory.",
        "This procedure provides instructions for evaluating sterility test samples under controlled conditions."
    ],
    "endotoxin": [
        "This procedure describes the method for bacterial endotoxin testing of product samples.",
        "This procedure defines the workflow for endotoxin analysis using validated laboratory practices.",
        "This procedure provides instructions for endotoxin test preparation and result recording."
    ],
    "bioburden": [
        "This procedure describes the method for bioburden testing of in-process or bulk samples.",
        "This procedure defines the steps required to estimate microbial load in collected samples.",
        "This procedure provides instructions for routine bioburden analysis in QC."
    ],
    "environmental": [
        "This procedure describes environmental monitoring activities in classified laboratory areas.",
        "This procedure defines the method for routine monitoring of controlled production and QC environments.",
        "This procedure provides instructions for viable and non-viable environmental monitoring."
    ],
    "buffer": [
        "This procedure describes the preparation of laboratory buffer solutions for routine analytical use.",
        "This procedure defines the steps for preparing and labeling synthetic buffer solutions.",
        "This procedure provides instructions for standardized buffer preparation in QC operations."
    ],
    "sample_handling": [
        "This procedure describes the handling, labeling, and storage of incoming QC samples.",
        "This procedure defines the requirements for sample receipt, transfer, and temporary storage.",
        "This procedure provides instructions for controlled handling of laboratory samples."
    ],
    "hplc": [
        "This procedure describes the preparation and execution of HPLC analysis for QC testing.",
        "This procedure defines the method for chromatographic sample preparation and system operation.",
        "This procedure provides instructions for routine HPLC testing in regulated laboratory work."
    ],
    "calibration": [
        "This procedure describes the calibration workflow for selected laboratory instruments.",
        "This procedure defines the requirements for routine calibration activities in QC.",
        "This procedure provides instructions for calibration checks and documentation of laboratory equipment."
    ]
}

SCOPE_LINES = [
    "This procedure applies to QC laboratory personnel.",
    "This procedure applies to analysts performing regulated laboratory tasks.",
    "This procedure applies to routine testing activities in GMP environments.",
    "This procedure applies to personnel assigned to controlled analytical operations."
]

RESPONSIBILITY_LINES = [
    "QC analysts are responsible for performing the procedure as written.",
    "Supervisors are responsible for reviewing completed records and confirming compliance.",
    "Laboratory personnel shall follow this procedure and document all relevant observations.",
    "Assigned analysts are responsible for execution, while supervisors are responsible for review."
]

DOCUMENTATION_LINES = [
    "All results shall be recorded in the controlled laboratory record form.",
    "All observations shall be documented immediately after completion of the task.",
    "Completed records shall be reviewed and archived according to site documentation procedures.",
    "All generated data shall be reviewed for completeness and retained in the approved archive."
]

# --------------------------------------------------
# SOP type-specific pools
# --------------------------------------------------

SOP_LIBRARY = {
    "sterility": {
        "titles": [
            "Sterility Testing Procedure for Final Product Samples",
            "Sterility Testing Procedure for Aseptic QC Samples",
            "Routine Sterility Testing Procedure",
            "Sterility Test Procedure for Product Release Samples"
        ],
        "materials": [
            "Sterility test canisters",
            "Membrane filtration unit",
            "Incubator",
            "Sterile forceps",
            "Growth medium bottles",
            "Sterile sample containers",
            "Alcohol spray",
            "Laminar flow hood"
        ],
        "steps": [
            "Verify that the sterility test area has been cleaned and released for use.",
            "Transfer the sample aseptically into the test setup.",
            "Filter the sample using the designated sterile membrane system.",
            "Rinse the membrane using sterile fluid as specified in the test plan.",
            "Transfer the membrane into the assigned growth medium.",
            "Incubate the test units for {incubation_days} days at the designated temperature range.",
            "Inspect the units daily for evidence of microbial growth.",
            "Record the incubation start time, end time, and all observations.",
            "Escalate any suspected contamination event to the supervisor immediately."
        ]
    },
    "endotoxin": {
        "titles": [
            "Endotoxin Test Procedure for QC Samples",
            "Bacterial Endotoxin Testing Procedure",
            "Routine Endotoxin Analysis Procedure",
            "Endotoxin Screening Procedure for Product Samples"
        ],
        "materials": [
            "Microplate reader",
            "Endotoxin reagent vials",
            "Pipettes",
            "Pyrogen-free tubes",
            "Disposable tips",
            "Vortex mixer",
            "Heating block",
            "Reagent water"
        ],
        "steps": [
            "Confirm that all endotoxin test reagents are within their assigned expiry period.",
            "Prepare the sample dilution using pyrogen-free water.",
            "Label each reaction tube with the assigned sample identifier.",
            "Add the designated reagent volume to each prepared tube.",
            "Mix the prepared solution for {mix_time} seconds.",
            "Incubate the reaction mixture at {incubation_temp}°C for the required reaction period.",
            "Measure the response using the designated instrument.",
            "Compare the measured value against the predefined acceptance criterion.",
            "Document all calculated and observed values in the approved worksheet."
        ]
    },
    "bioburden": {
        "titles": [
            "Bioburden Sampling Procedure",
            "Routine Bioburden Test Procedure",
            "Bioburden Analysis Procedure for In-Process Samples",
            "Microbial Load Testing Procedure"
        ],
        "materials": [
            "Sterile sample bottles",
            "Membrane filters",
            "Vacuum manifold",
            "Incubator",
            "Agar plates",
            "Dilution tubes",
            "Sterile saline",
            "Marker labels"
        ],
        "steps": [
            "Verify sample identity before initiating the bioburden workflow.",
            "Prepare the required dilution series using sterile diluent.",
            "Transfer the assigned sample volume to the test vessel.",
            "Filter the prepared sample through the designated membrane.",
            "Place the membrane onto the selected agar medium.",
            "Incubate the plates for {incubation_days} days under defined culture conditions.",
            "Count visible colonies after the incubation period.",
            "Calculate the microbial load per defined sample volume.",
            "Document any atypical colony morphology in the record form."
        ]
    },
    "environmental": {
        "titles": [
            "Environmental Monitoring Procedure",
            "Routine Environmental Monitoring Procedure for Controlled Areas",
            "Environmental Monitoring Sampling Procedure",
            "Microbiological Environmental Monitoring Procedure"
        ],
        "materials": [
            "Settle plates",
            "Active air sampler",
            "Contact plates",
            "Sampling map",
            "Marker pen",
            "Alcohol wipes",
            "Incubator",
            "Sampling log sheet"
        ],
        "steps": [
            "Verify the sampling plan and room classification before monitoring.",
            "Place the settle plates at the designated monitoring locations.",
            "Perform active air sampling for {sampling_time} minutes at the required location.",
            "Collect surface samples using contact plates according to the site sampling map.",
            "Label all collected media with area, date, and sample point.",
            "Transfer collected media to the incubator under controlled conditions.",
            "Incubate samples according to the approved environmental monitoring plan.",
            "Record microbial growth observations after incubation.",
            "Escalate any excursion above the action limit to Quality Assurance."
        ]
    },
    "buffer": {
        "titles": [
            "Phosphate Buffer Preparation Procedure",
            "Citrate Buffer Preparation Procedure",
            "Wash Buffer Preparation Procedure",
            "Mobile Phase Preparation Procedure",
            "Acetate Buffer Preparation Procedure",
            "Tris Buffer Preparation Procedure"
        ],
        "materials": [
            "Analytical balance",
            "pH meter",
            "Magnetic stirrer",
            "Beaker",
            "Graduated cylinder",
            "Purified water",
            "Buffer salt",
            "Label printer"
        ],
        "steps": [
            "Verify the identity and status of all required raw materials before preparation.",
            "Measure {buffer_mass} g of the designated buffer component.",
            "Add the measured material to {buffer_volume} mL of purified water.",
            "Mix the solution at {mix_speed} rpm until the material is dissolved.",
            "Adjust the solution to pH {target_ph} using acid or base as required.",
            "Transfer the prepared solution into the assigned storage container.",
            "Label the container with solution name, preparation date, and expiry.",
            "Store the prepared buffer at {storage_temp}°C unless otherwise specified.",
            "Document preparation details in the approved preparation record."
        ]
    },
    "sample_handling": {
        "titles": [
            "Sample Handling Procedure",
            "Sample Receipt and Handling Procedure",
            "QC Sample Transfer Procedure",
            "Sample Labeling and Storage Procedure"
        ],
        "materials": [
            "Sample logbook",
            "Barcode labels",
            "Insulated transfer box",
            "Sample containers",
            "Temperature monitor",
            "Permanent marker",
            "Receiving checklist",
            "Storage rack"
        ],
        "steps": [
            "Confirm sample identity against the accompanying documentation.",
            "Inspect the sample container for integrity and labeling accuracy.",
            "Assign the sample tracking number using the approved log system.",
            "Attach a barcode or manual label to the received sample.",
            "Transfer the sample to the designated storage area within {transfer_time} minutes.",
            "Store the sample at {storage_temp}°C according to handling requirements.",
            "Record receipt time, condition, and storage location in the log.",
            "Notify the responsible analyst that the sample is available for testing.",
            "Escalate any discrepancy in labeling or condition to the supervisor."
        ]
    },
    "hplc": {
        "titles": [
            "Protein A HPLC Procedure",
            "Routine HPLC Analysis Procedure",
            "Chromatographic Sample Preparation and Analysis Procedure",
            "HPLC System Use Procedure for QC Testing"
        ],
        "materials": [
            "HPLC system",
            "Autosampler vials",
            "Mobile phase bottles",
            "Analytical column",
            "Syringe filters",
            "Pipettes",
            "Integration software",
            "Waste bottle"
        ],
        "steps": [
            "Verify instrument status and ensure that calibration is current before use.",
            "Prepare the mobile phase according to the approved composition.",
            "Filter the sample using a {filter_size} µm syringe filter.",
            "Transfer the filtered sample into a labeled autosampler vial.",
            "Set the column temperature to {column_temp}°C.",
            "Run the chromatographic method using the approved sequence table.",
            "Review peak shape, retention time, and integration suitability.",
            "Compare the result against the predefined system suitability criteria.",
            "Document all observations and calculated values in the approved worksheet."
        ]
    },
    "calibration": {
        "titles": [
            "pH Meter Calibration Procedure",
            "Analytical Balance Calibration Check Procedure",
            "Routine Instrument Calibration Procedure",
            "Laboratory Equipment Calibration Verification Procedure"
        ],
        "materials": [
            "Calibration standard",
            "Reference weight set",
            "Calibration logbook",
            "Instrument label",
            "Cleaning wipes",
            "Temperature probe",
            "Adjustment tool",
            "Approved checklist"
        ],
        "steps": [
            "Verify instrument identification and current status before calibration.",
            "Ensure the instrument is clean and ready for calibration activity.",
            "Prepare the required standard or reference material.",
            "Perform the calibration sequence according to the approved instruction set.",
            "Record observed values at each calibration point.",
            "Verify that all results fall within the defined tolerance of ±{tolerance}.",
            "Apply a calibration status label after completion of the activity.",
            "Document the calibration outcome in the approved equipment record.",
            "Escalate any out-of-tolerance condition according to site procedure."
        ]
    }
}

# --------------------------------------------------
# Parameter generators
# --------------------------------------------------

def random_parameters():
    return {
        "incubation_days": random.choice([3, 5, 7, 10, 14]),
        "mix_time": random.choice([10, 15, 20, 30, 45, 60]),
        "incubation_temp": random.choice([30, 35, 37]),
        "sampling_time": random.choice([2, 4, 5, 10, 15]),
        "buffer_mass": random.choice([2.5, 4.0, 5.5, 8.0, 10.0, 12.5]),
        "buffer_volume": random.choice([250, 500, 1000, 2000]),
        "mix_speed": random.choice([150, 200, 250, 300, 400]),
        "target_ph": random.choice([5.0, 5.5, 6.8, 7.0, 7.2, 7.4]),
        "storage_temp": random.choice(["2–8", "15–25", "20–25"]),
        "transfer_time": random.choice([15, 20, 30, 45, 60]),
        "filter_size": random.choice([0.2, 0.22, 0.45]),
        "column_temp": random.choice([25, 30, 35, 40]),
        "tolerance": random.choice(["0.05", "0.1", "0.2", "0.5"])
    }

# --------------------------------------------------
# PDF generation
# --------------------------------------------------

def draw_wrapped_line(c, text, x, y, max_chars=90, line_height=15):
    words = text.split()
    line = ""

    for word in words:
        trial = f"{line} {word}".strip()
        if len(trial) <= max_chars:
            line = trial
        else:
            c.drawString(x, y, line)
            y -= line_height
            line = word

    if line:
        c.drawString(x, y, line)
        y -= line_height

    return y


def create_sop_pdf(file_path: str, sop_type: str, title: str) -> None:
    c = canvas.Canvas(file_path, pagesize=letter)
    _, height = letter
    y = height - 50

    params = random_parameters()
    lib = SOP_LIBRARY[sop_type]

    c.setFont("Helvetica-Bold", 16)
    c.drawString(50, y, title)
    y -= 35

    sections = [
        ("1. Purpose", random.choice(PURPOSE_LINES[sop_type])),
        ("2. Scope", random.choice(SCOPE_LINES)),
        ("3. Responsibilities", random.choice(RESPONSIBILITY_LINES)),
        ("4. Materials and Equipment", None),
        ("5. Procedure", None),
        ("6. Documentation", random.choice(DOCUMENTATION_LINES)),
    ]

    for section_title, section_body in sections:
        if y < 120:
            c.showPage()
            y = height - 50

        c.setFont("Helvetica-Bold", 12)
        c.drawString(50, y, section_title)
        y -= 20

        c.setFont("Helvetica", 11)

        if section_title == "4. Materials and Equipment":
            selected_materials = random.sample(lib["materials"], k=min(5, len(lib["materials"])))
            for item in selected_materials:
                y = draw_wrapped_line(c, f"- {item}", 70, y)
                y -= 2

        elif section_title == "5. Procedure":
            selected_steps = random.sample(lib["steps"], k=min(7, len(lib["steps"])))
            for i, step in enumerate(selected_steps, start=1):
                formatted_step = step.format(**params)
                y = draw_wrapped_line(c, f"5.{i} {formatted_step}", 70, y)
                y -= 2

        else:
            y = draw_wrapped_line(c, section_body, 70, y)
            y -= 4

        y -= 8

    c.save()

# --------------------------------------------------
# Main generator
# --------------------------------------------------

def generate_sops():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    sop_types = list(SOP_LIBRARY.keys())

    for i in range(1, NUM_FILES + 1):
        sop_type = random.choice(sop_types)
        title = random.choice(SOP_LIBRARY[sop_type]["titles"])

        file_name = f"QC_SOP_{i:03}.pdf"
        file_path = os.path.join(OUTPUT_DIR, file_name)

        create_sop_pdf(file_path, sop_type, title)

    print(f"{NUM_FILES} synthetic SOP PDFs created in: {OUTPUT_DIR}")

# --------------------------------------------------
# Entry point
# --------------------------------------------------

if __name__ == "__main__":
    generate_sops()

500 synthetic SOP PDFs created in: synthetic_sops/qc


In [30]:
pip install pdfplumber pandas

Note: you may need to restart the kernel to use updated packages.


In [31]:
import pdfplumber
import pandas as pd

In [32]:
from pathlib import Path

pdf_files = sorted(Path("synthetic_sops/qc").glob("*.pdf"))
len(pdf_files), pdf_files[:3]

(500,
 [WindowsPath('synthetic_sops/qc/QC_SOP_001.pdf'),
  WindowsPath('synthetic_sops/qc/QC_SOP_002.pdf'),
  WindowsPath('synthetic_sops/qc/QC_SOP_003.pdf')])

In [33]:
import pdfplumber

with pdfplumber.open("synthetic_sops/qc/QC_SOP_001.pdf") as pdf:
    
    page = pdf.pages[0]
    
    text = page.extract_text()
    
    print(text)

Bioburden Analysis Procedure for In-Process Samples
1. Purpose
This procedure defines the steps required to estimate microbial load in collected samples.
2. Scope
This procedure applies to analysts performing regulated laboratory tasks.
3. Responsibilities
QC analysts are responsible for performing the procedure as written.
4. Materials and Equipment
- Marker labels
- Incubator
- Membrane filters
- Agar plates
- Sterile saline
5. Procedure
5.1 Count visible colonies after the incubation period.
5.2 Filter the prepared sample through the designated membrane.
5.3 Document any atypical colony morphology in the record form.
5.4 Place the membrane onto the selected agar medium.
5.5 Prepare the required dilution series using sterile diluent.
5.6 Incubate the plates for 7 days under defined culture conditions.
5.7 Verify sample identity before initiating the bioburden workflow.
6. Documentation
All generated data shall be reviewed for completeness and retained in the approved
archive.


In [34]:
import pdfplumber
from pathlib import Path

pdf_files = sorted(Path("synthetic_sops/qc").glob("*.pdf"))

records = []

for pdf_file in pdf_files:
    
    with pdfplumber.open(pdf_file) as pdf:
        
        for page_number, page in enumerate(pdf.pages, start=1):
            
            text = page.extract_text()
            
            records.append({
                "sop_id": pdf_file.stem,
                "page_number": page_number,
                "text": text
            })

len(records)

500

In [35]:
from pathlib import Path

pdf_files = sorted(Path(r"C:\Users\user\synthetic_sops\qc").glob("*.pdf"))
len(pdf_files), pdf_files[:3], pdf_files[-3:]

(500,
 [WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_001.pdf'),
  WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_002.pdf'),
  WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_003.pdf')],
 [WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_498.pdf'),
  WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_499.pdf'),
  WindowsPath('C:/Users/user/synthetic_sops/qc/QC_SOP_500.pdf')])

In [36]:
import pdfplumber
from pathlib import Path

pdf_files = sorted(Path(r"C:\Users\user\synthetic_sops\qc").glob("*.pdf"))

records = []

for pdf_file in pdf_files:
    
    with pdfplumber.open(pdf_file) as pdf:
        
        for page_number, page in enumerate(pdf.pages, start=1):
            
            text = page.extract_text()
            
            records.append({
                "sop_id": pdf_file.stem,
                "page_number": page_number,
                "text": text
            })

len(records)

500

In [37]:
import pandas as pd

df = pd.DataFrame(records)

df.head()

,sop_id,page_number,text
0,QC_SOP_001,1,Bioburden Analysis Procedure for In-Process Sa...
1,QC_SOP_002,1,Routine Endotoxin Analysis Procedure\n1. Purpo...
2,QC_SOP_003,1,Endotoxin Screening Procedure for Product Samp...
3,QC_SOP_004,1,Sample Labeling and Storage Procedure\n1. Purp...
4,QC_SOP_005,1,Bioburden Sampling Procedure\n1. Purpose\nThis...


In [38]:
df.to_csv("parsed_pages.csv", index=False)

In [39]:
import re

keywords = []

for _, row in df.iterrows():
    
    words = re.findall(r"[A-Za-z]+", row["text"].lower())
    
    unique_words = set(words)
    
    for w in unique_words:
        
        if len(w) > 6:  
            keywords.append({
                "keyword": w,
                "sop_id": row["sop_id"],
                "page": row["page_number"]
            })

len(keywords)

24530

In [40]:
keyword_df = pd.DataFrame(keywords)
keyword_df.head()

,keyword,sop_id,page
0,colonies,QC_SOP_001,1
1,regulated,QC_SOP_001,1
2,estimate,QC_SOP_001,1
3,archive,QC_SOP_001,1
4,initiating,QC_SOP_001,1


In [41]:
keyword_df.to_csv("keyword_index.csv", index=False)

In [42]:
search_term = "sterility"

results = keyword_df[keyword_df["keyword"] == search_term]

results.head(10)

,keyword,sop_id,page
722,sterility,QC_SOP_015,1
2529,sterility,QC_SOP_052,1
2632,sterility,QC_SOP_054,1
2974,sterility,QC_SOP_061,1
3073,sterility,QC_SOP_063,1
3470,sterility,QC_SOP_071,1
3524,sterility,QC_SOP_072,1
4171,sterility,QC_SOP_085,1
4316,sterility,QC_SOP_088,1
4415,sterility,QC_SOP_090,1


In [43]:
search_term = "sterility"

results = keyword_df[keyword_df["keyword"] == search_term]

merged_results = results.merge(
    df,
    left_on=["sop_id", "page"],
    right_on=["sop_id", "page_number"],
    how="left"
)

merged_results[["keyword", "sop_id", "page", "text"]].head(5)

,keyword,sop_id,page,text
0,sterility,QC_SOP_015,1,Sterility Testing Procedure for Aseptic QC Sam...
1,sterility,QC_SOP_052,1,Sterility Test Procedure for Product Release S...
2,sterility,QC_SOP_054,1,Sterility Testing Procedure for Final Product ...
3,sterility,QC_SOP_061,1,Sterility Testing Procedure for Final Product ...
4,sterility,QC_SOP_063,1,Sterility Test Procedure for Product Release S...


In [44]:
merged_results["snippet"] = merged_results["text"].str[:120]

merged_results[["keyword","sop_id","page","snippet"]].head(5)

,keyword,sop_id,page,snippet
0,sterility,QC_SOP_015,1,Sterility Testing Procedure for Aseptic QC Sam...
1,sterility,QC_SOP_052,1,Sterility Test Procedure for Product Release S...
2,sterility,QC_SOP_054,1,Sterility Testing Procedure for Final Product ...
3,sterility,QC_SOP_061,1,Sterility Testing Procedure for Final Product ...
4,sterility,QC_SOP_063,1,Sterility Test Procedure for Product Release S...


In [45]:
merged_results["clean_text"] = (
    merged_results["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

merged_results["snippet"] = merged_results["clean_text"].str[:120] + "..."

merged_results[["keyword", "sop_id", "page", "snippet"]].head(5)

,keyword,sop_id,page,snippet
0,sterility,QC_SOP_015,1,Sterility Testing Procedure for Aseptic QC Sam...
1,sterility,QC_SOP_052,1,Sterility Test Procedure for Product Release S...
2,sterility,QC_SOP_054,1,Sterility Testing Procedure for Final Product ...
3,sterility,QC_SOP_061,1,Sterility Testing Procedure for Final Product ...
4,sterility,QC_SOP_063,1,Sterility Test Procedure for Product Release S...


In [46]:
def search_sop(keyword):

    results = keyword_df[keyword_df["keyword"] == keyword]

    merged = results.merge(
        df,
        left_on=["sop_id", "page"],
        right_on=["sop_id", "page_number"],
        how="left"
    )

    merged["clean_text"] = (
        merged["text"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    merged["snippet"] = merged["clean_text"].str[:120] + "..."

    return merged[["sop_id","page","snippet"]].head(10)

In [47]:
search_sop("sterility")

,sop_id,page,snippet
0,QC_SOP_015,1,Sterility Testing Procedure for Aseptic QC Sam...
1,QC_SOP_052,1,Sterility Test Procedure for Product Release S...
2,QC_SOP_054,1,Sterility Testing Procedure for Final Product ...
3,QC_SOP_061,1,Sterility Testing Procedure for Final Product ...
4,QC_SOP_063,1,Sterility Test Procedure for Product Release S...
5,QC_SOP_071,1,Routine Sterility Testing Procedure 1. Purpose...
6,QC_SOP_072,1,Sterility Test Procedure for Product Release S...
7,QC_SOP_085,1,Sterility Test Procedure for Product Release S...
8,QC_SOP_088,1,Sterility Test Procedure for Product Release S...
9,QC_SOP_090,1,Sterility Testing Procedure for Aseptic QC Sam...


In [28]:
!pip install streamlit

In [48]:
pdf_files = sorted(Path("synthetic_sops/qc").glob("*.pdf"))

In [49]:
df = pd.DataFrame(records)

In [50]:
df.to_csv("parsed_pages.csv", index=False)

In [51]:
keyword_df.to_csv("keyword_index.csv", index=False)

In [52]:
df[df["sop_id"] == "QC_SOP_026"][["sop_id", "text"]].head(1)

,sop_id,text
25,QC_SOP_026,Sample Receipt and Handling Procedure\n1. Purp...


In [53]:
keyword_df[keyword_df["keyword"].str.contains("buffer", case=False, na=False)].head(20)

,keyword,sop_id,page


In [54]:
keyword_df[keyword_df["keyword"].str.contains("buffer", case=False, na=False)].head(20)

,keyword,sop_id,page
